# The Sum of Expectations

When you add random variables, their expected values add: $E[X + Y] = E[X] + E[Y]$. That sounds almost too simple to be useful — but it holds for *any* distributions, **whether or not the variables are independent**, and it cracks problems that look impossibly hard. The showpiece: hand every person on Earth a random name from a bag of all 8 billion names. How many people get their own name, on average? Exactly **one**. This notebook shows why. It is based on the fourth video of Lesson 3 (Describing Distributions).

## 1. Introduction

You know how to compute the expected value of one random variable, and of a function of it. Real situations usually involve *several* sources of randomness at once: a game with a coin flip *and* a die roll, a portfolio of many assets, a sequence of many trials.

Why this matters:

- Computing the distribution of a **sum** of random variables is usually hard; computing the expectation of the sum is trivial — it is just the sum of the expectations.
- The rule needs **no independence**. That makes it one of the most broadly applicable tools in probability.
- The technique of splitting a complicated count into many tiny 0/1 pieces ("did person $i$ get their name?") turns intractable averages into one-line calculations.

After completing this notebook you will be able to:

- Use $E[X + Y] = E[X] + E[Y]$ and its $n$-variable version $E[X_1 + \cdots + X_n] = \sum_i E[X_i]$.
- Explain why the rule holds even for dependent variables.
- Solve the name-matching problem by decomposing a count into per-person contributions.
- Recognize when a hard "expected count" problem becomes easy with this decomposition.

## 2. Theory

### A game with two parts

Play two things in one game:

1. Flip a coin — heads wins **`$1`**, tails wins nothing.
2. Roll a die — win the amount you roll.

Call the first payoff $X_{\text{coin}}$ and the second $X_{\text{dice}}$. You already know both means:

$$
E[X_{\text{coin}}] = 0.50, \qquad E[X_{\text{dice}}] = 3.50
$$

The total payoff is the *sum* $X = X_{\text{coin}} + X_{\text{dice}}$, and its expected value is simply

$$
E[X] = E[X_{\text{coin}}] + E[X_{\text{dice}}] = 0.50 + 3.50 = 4
$$

### The general rule

For any two random variables:

$$
E[X + Y] = E[X] + E[Y]
$$

and for any number of them:

$$
E[X_1 + X_2 + \cdots + X_n] = E[X_1] + E[X_2] + \cdots + E[X_n]
$$

This result is **always valid**: it does not depend on the distributions of the variables, and it does not matter whether they are independent or not. (Contrast this with probabilities of joint events, where independence mattered a lot!)

Together with the last notebook's $E[aX + b] = aE[X] + b$, this completes the picture of the expectation as a **linear operator**.

### Why it is powerful: decompose a hard count

The trick that makes this rule shine: when a quantity is a complicated **count**, write it as a sum of simple 0/1 variables — one per item, equal to 1 when the item contributes and 0 otherwise. Each little variable has a trivial expectation (just a probability!), and the expectations add — no matter how tangled the dependencies between the items are.

## 3. Worked Examples

### Example 1: the coin-and-die game

As above: $E[X_{\text{coin}} + X_{\text{dice}}] = 0.5 + 3.5 = 4$ per play, on average.

### Example 2: the name game, brute force ($n = 3$)

A bag holds the names of Aisha, Beto, and Cameron. Hand each of the three a random name from the bag. How many people get their **own** name, on average?

There are $3! = 6$ equally likely ways to hand out the names. Count the correct assignments in each:

| assignment (Aisha, Beto, Cameron get…) | matches |
|---|---|
| Aisha, Beto, Cameron | 3 |
| Aisha, Cameron, Beto | 1 |
| Beto, Aisha, Cameron | 1 |
| Beto, Cameron, Aisha | 0 |
| Cameron, Aisha, Beto | 0 |
| Cameron, Beto, Aisha | 1 |

Average: $\frac{3 + 1 + 1 + 0 + 0 + 1}{6} = \frac{6}{6} = 1$. The expected number of matches is **1**.

### Example 3: the name game, the clever way

Brute force dies quickly — for 8 billion people there are $n!$ assignments. Instead, split the count. Let

- $A$ = number of matches involving Aisha (0 or 1),
- $B$ = the same for Beto, $C$ for Cameron,

so $\text{Matches} = A + B + C$. Aisha receives each of the 3 names with probability $\tfrac13$, so she gets her own with probability $\tfrac13$ and $E[A] = \tfrac13$. Likewise $E[B] = E[C] = \tfrac13$. Therefore

$$
E[\text{Matches}] = E[A] + E[B] + E[C] = \tfrac13 + \tfrac13 + \tfrac13 = 1
$$

No enumeration needed. Note that $A$, $B$, $C$ are *not* independent (if Aisha and Beto both got their names, Cameron must have too) — and the rule does not care.

### Example 4: the name game with all 8 billion people

Persons $1, \dots, n$ with $n = 8{,}000{,}000{,}000$:

$$
E[\text{Matches}] = E[X_{\text{person}_1}] + \cdots + E[X_{\text{person}_n}]
 = \underbrace{\tfrac{1}{n} + \tfrac{1}{n} + \cdots + \tfrac{1}{n}}_{n \text{ times}} = n \cdot \tfrac{1}{n} = 1
$$

Each person gets their own name with probability $\tfrac1n$, and $n$ of those tiny expectations sum to exactly **1** — for three people, for 8 billion, for any $n$.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import permutations

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

### The coin-and-die game: expectations add

In [ ]:
rng = np.random.default_rng(42)

n_plays = 100_000
coin_payoff = rng.choice([1, 0], size=n_plays)      # heads -> $1
dice_payoff = rng.integers(1, 7, size=n_plays)      # win the roll
total_payoff = coin_payoff + dice_payoff

print(f"E[X_coin]  (simulated): {coin_payoff.mean():.4f}   (theory 0.5)")
print(f"E[X_dice]  (simulated): {dice_payoff.mean():.4f}   (theory 3.5)")
print(f"E[X_total] (simulated): {total_payoff.mean():.4f}   (theory 0.5 + 3.5 = 4.0)")

### The name game for $n = 3$: enumerate all 6 assignments

A random handout of names is a random *permutation*; a match is a position where the permutation keeps its element in place.

In [ ]:
people = ["Aisha", "Beto", "Cameron"]

total_matches = 0
print(f"{'assignment':<28} matches")
for assignment in permutations(people):
    matches = sum(given == owner for given, owner in zip(assignment, people))
    total_matches += matches
    print(f"{str(assignment):<28} {matches}")

n_assignments = 6
print(f"\nExpected matches = {total_matches}/{n_assignments} = {total_matches / n_assignments}")

### The decomposition view: each person contributes $1/n$

Simulate the game and track each person separately — every one of them matches about a third of the time, and the per-person rates sum to the total.

In [ ]:
rng = np.random.default_rng(42)

n_games = 100_000
n = 3
# each row is one game: a random permutation of the names 0, 1, 2
handouts = np.array([rng.permutation(n) for _ in range(n_games)])
matches_per_person = handouts == np.arange(n)       # True where person i got name i

for i, name in enumerate(["Aisha", "Beto", "Cameron"]):
    print(f"E[{name:<7}] (simulated): {matches_per_person[:, i].mean():.4f}   (theory 1/3 = 0.3333)")
print(f"E[Matches]  (simulated): {matches_per_person.sum(axis=1).mean():.4f}   (theory 1)")

### The expected number of matches is 1 for every $n$

We cannot simulate 8 billion people, but we can sweep $n$ over several orders of magnitude — the expected match count stays pinned at 1.

In [ ]:
rng = np.random.default_rng(42)

print(f"{'n':>8} {'expected matches (simulated)':>30}")
for n in [2, 3, 10, 100, 1000, 10_000]:
    n_games = 200_000 // n + 2000          # fewer games for bigger n, still accurate
    total = 0
    for _ in range(n_games):
        total += np.sum(rng.permutation(n) == np.arange(n))
    print(f"{n:>8} {total / n_games:>30.4f}")
print("\nTheory: n * (1/n) = 1 for every n -- including n = 8,000,000,000.")

### Dependence does not matter

Take $X$ = one die roll and $Y = 7 - X$ — completely dependent (knowing $X$ fixes $Y$). The expectations still add: $E[X + Y] = 3.5 + 3.5 = 7$, exactly (the sum is *always* 7 here).

In [ ]:
rng = np.random.default_rng(42)

x = rng.integers(1, 7, size=100_000)
y = 7 - x                                  # totally dependent on x

print(f"E[X]     = {x.mean():.4f}")
print(f"E[Y]     = {y.mean():.4f}")
print(f"E[X + Y] = {(x + y).mean():.4f}   = E[X] + E[Y], despite total dependence")

## 5. Visualizations

### The two-part game: the sum's distribution balances at 4

The total payoff takes values 1–7 with a shape that is neither the coin's nor the die's — but its balance point is just $0.5 + 3.5$.

In [ ]:
rng = np.random.default_rng(42)

coin = rng.choice([1, 0], size=100_000)
dice = rng.integers(1, 7, size=100_000)
total = coin + dice

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, data, title, mean_theory in [
        (axes[0], coin, "coin payoff", 0.5),
        (axes[1], dice, "die payoff", 3.5),
        (axes[2], total, "total payoff", 4.0)]:
    values, counts = np.unique(data, return_counts=True)
    ax.bar(values, counts / len(data), color="tab:blue", edgecolor="black", alpha=0.85)
    ax.axvline(data.mean(), color="tab:red", linestyle="--", linewidth=2,
               label=f"mean = {data.mean():.2f}")
    ax.set_title(f"{title}  (theory E = {mean_theory})")
    ax.set_xlabel("payoff ($)")
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("relative frequency")
plt.suptitle("E[coin + die] = E[coin] + E[die] = 0.5 + 3.5 = 4")
plt.tight_layout()
plt.show()

### The name game: the distribution of matches, $n = 3$

Out of the 6 assignments: three give 1 match, two give 0, one gives 3 (and 2 matches is impossible — if two people have their own names, so does the third!). The balance point is 1.

In [ ]:
match_values = np.array([0, 1, 2, 3])
match_probs = np.array([2, 3, 0, 1]) / 6

plt.figure(figsize=(8, 5))
plt.bar(match_values, match_probs, color="tab:blue", edgecolor="black", alpha=0.85)
plt.axvline(np.sum(match_values * match_probs), color="tab:red", linestyle="--",
            linewidth=2, label="E[Matches] = 1")
plt.annotate("2 matches is impossible:\nif two are right, all three are",
             xy=(2, 0.005), xytext=(1.9, 0.2),
             arrowprops=dict(arrowstyle="->"), fontsize=10)
plt.title("Number of correct assignments for 3 people")
plt.xlabel("number of matches")
plt.ylabel("probability")
plt.xticks(match_values)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The expected match count stays at 1 as $n$ grows

The *distribution* of matches changes with $n$, but its mean never moves — every extra person adds a $\tfrac1n$ chance across $n$ people.

In [ ]:
rng = np.random.default_rng(42)

ns = [2, 3, 5, 10, 30, 100, 300, 1000]
means = []
for n in ns:
    n_games = 5000
    total = sum(np.sum(rng.permutation(n) == np.arange(n)) for _ in range(n_games))
    means.append(total / n_games)

plt.figure(figsize=(9, 5))
plt.plot(ns, means, "o-", color="tab:blue", label="simulated E[Matches]")
plt.axhline(1, color="tab:red", linestyle="--", label="theory: n · (1/n) = 1")
plt.xscale("log")
plt.title("Expected number of matches vs. number of people")
plt.xlabel("number of people n (log scale)")
plt.ylabel("expected matches")
plt.ylim(0, 2)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The whole story on one page

Everything above in a single sheet: the coin-and-die game, all six name-game handouts with their per-person averages, the distribution of matches, and why the expected count stays at 1 for every n.

<img src="img/04_sum_of_expectations.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** You roll two fair dice and win the sum. What is the expected win? (One line — no enumeration of the 36 outcomes.)

**Exercise 2.** A game pays the coin flip (`$1` for heads) **plus** twice a die roll **minus** a `$3` entry fee. Combine the sum rule with linearity to find the expected net win.

### Intermediate

**Exercise 3.** You flip 100 fair coins. Using the decomposition "total heads $=$ sum of 100 per-coin indicator variables," find the expected number of heads — without ever writing the binomial PMF. Verify by simulation.

**Exercise 4.** Roll one die and let $X$ be the result; let $Y = X$ (the *same* roll, maximally dependent). What are $E[X + Y]$ and $E[X \cdot Y]$? Which of the two can you get by just adding/multiplying $E[X]$ — and why does the other fail?

### Advanced

**Exercise 5.** In a room of 23 people, define one indicator variable per *pair* that equals 1 when the two share a birthday. Use the sum of expectations to compute the expected number of shared-birthday **pairs** ($\binom{23}{2}$ pairs, each matching with probability $\tfrac{1}{365}$). Verify by simulation. How does this connect to the birthday problem's famous "23"?

**Exercise 6.** For the name game, the expected number of matches is 1 for every $n$ — but what does the *distribution* of matches look like for large $n$? Simulate $n = 100$ and compare the histogram with the Poisson(1) probabilities $P(k) = \frac{e^{-1}}{k!}$. What is the probability that *nobody* gets their own name?

## 7. Solutions

In [ ]:
# Exercise 1: sum of two dice
print(f"Exercise 1: E[die1 + die2] = 3.5 + 3.5 = {3.5 + 3.5}")

In [ ]:
# Exercise 2: combine the sum rule with linearity
# E[coin + 2*die - 3] = E[coin] + 2*E[die] - 3
expected = 0.5 + 2 * 3.5 - 3
print(f"Exercise 2: E = 0.5 + 2*3.5 - 3 = {expected}")

In [ ]:
# Exercise 3: 100 coins via indicators
rng = np.random.default_rng(0)

# Each coin i contributes an indicator with E = 0.5; the sum rule gives 100 * 0.5.
theory = 100 * 0.5
simulated = (rng.random(size=(100_000, 100)) < 0.5).sum(axis=1).mean()
print(f"Exercise 3: theory = {theory}, simulated = {simulated:.3f}")
print("No binomial PMF needed -- just 100 indicators, each with expectation 1/2.")

In [ ]:
# Exercise 4: same roll twice
rng = np.random.default_rng(0)
x = rng.integers(1, 7, size=100_000)
y = x

print(f"Exercise 4: E[X + Y] = {(x + y).mean():.4f}  = 3.5 + 3.5 (sum rule, dependence irrelevant)")
print(f"            E[X * Y] = {(x * y).mean():.4f}  but E[X]*E[Y] = {3.5 * 3.5}")
print("Sums of expectations always add; PRODUCTS do not (unless independent --")
print("here X*Y = X^2, and E[X^2] = 91/6 = 15.17 from the last notebook).")

In [ ]:
# Exercise 5: expected shared-birthday pairs among 23 people
rng = np.random.default_rng(0)

n_people = 23
n_pairs = n_people * (n_people - 1) // 2
theory = n_pairs / 365

n_rooms = 100_000
birthdays = rng.integers(0, 365, size=(n_rooms, n_people))
pair_counts = np.zeros(n_rooms)
for i in range(n_people):
    for j in range(i + 1, n_people):
        pair_counts += birthdays[:, i] == birthdays[:, j]

print(f"Exercise 5: C(23,2) = {n_pairs} pairs, each matches w.p. 1/365")
print(f"Expected pairs: theory = {theory:.4f}, simulated = {pair_counts.mean():.4f}")
print("23 is the point where the EXPECTED pair count reaches ~0.69 -- of the same")
print("order as 1 -- which is why the chance of at least one match passes 50%.")

In [ ]:
# Exercise 6: distribution of matches for n = 100 vs Poisson(1)
from math import factorial, e
rng = np.random.default_rng(0)

n = 100
n_games = 50_000
matches = np.array([np.sum(rng.permutation(n) == np.arange(n)) for _ in range(n_games)])

print("Exercise 6:")
print(f"{'k':>3} {'simulated':>10} {'Poisson(1)':>11}")
for k in range(6):
    poisson = e**-1 / factorial(k)
    print(f"{k:>3} {np.mean(matches == k):>10.4f} {poisson:>11.4f}")
print(f"\nP(nobody gets their name) ~ 1/e = {1/e:.4f} -- and the mean stays {matches.mean():.3f}.")

## 8. Conclusion

- **Expectations add:** $E[X + Y] = E[X] + E[Y]$, and $E[X_1 + \cdots + X_n] = \sum_i E[X_i]$.
- The rule is *unconditional*: any distributions, **independent or not**. (Products of expectations enjoy no such rule.)
- The coin-and-die game prices at $0.5 + 3.5 = 4$ — no need to work out the distribution of the sum.
- The killer technique: **decompose a count into 0/1 indicator variables.** Each person gets their own name with probability $\tfrac1n$, so the expected number of matches is $n \cdot \tfrac1n = 1$ — for 3 people or 8 billion.
- Simple rule, non-trivial results: expected heads in 100 flips ($100 \cdot \tfrac12$), expected shared-birthday pairs ($\binom{23}{2}/365$), expected fixed points of a random permutation (always 1).

Next up: the **variance** — using expected values to describe the *spread* of a distribution.